# Lesson 11 Lab — Gradual Pruning Schedules and Recovery Training

**Puzzle:** What does a polynomial sparsity schedule control that a final target does not?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

A target sparsity says where training should end; a schedule says how abruptly the feasible parameter set changes. With a fixed 15% recovery budget, pruning events compete with optimization steps, so the begin step, end step, update frequency, and learning-rate trajectory become part of the result.


## 0. Predict before running

1. Compute the target sparsity halfway through a cubic schedule.
2. Predict which route has the largest immediate loss shock.
3. Name the schedule fields needed to reproduce a recovery trajectory.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

The experiment uses one frozen dense classifier, an immediate target mask, a polynomial target function, periodic magnitude-mask updates, identical recovery updates, and a held-out accuracy trajectory.

- Final sparsity does not identify the support trajectory.
- Pruning frequency trades adaptation time against ranking refresh.
- Learning-rate and sparsity schedules interact.


## 2. Derive the mechanism

A common cubic schedule is `s(t)=s_f+(s_i-s_f)(1-(t-t0)/(t1-t0))^3` within the pruning window. Early updates remove few weights and later changes taper as the target is approached. The schedule does not guarantee recovery; it bounds the size and timing of support shocks. Reapplying a newly ranked mask can remove previously useful weights, while a fixed mask only trains survivors. Those policy choices must be frozen.

Keep value sparsity, physical shape, representation, and runtime evidence separate.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 11
LESSON_TITLE = 'Gradual Pruning Schedules and Recovery Training'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260819
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | one-shot jump to 80% sparsity at the first recovery step |
| Candidate | cubic updates from 0% to 80% over the same recovery window |
| Held constant | dense checkpoint, data order, optimizer, learning rate, total steps, target rate, and mask rule |
| Measurements | target/actual sparsity trajectory, immediate loss, final accuracy, and best accuracy |
| Evidence | `pytorch-gpu` |

**Experiment:** Compare one-shot and cubic gradual schedules under one initialization and equal optimizer-step budget.


## 5. Read the experiment code

The notebook records a row at each schedule update rather than only the endpoint. Both routes clone the same baseline and execute the same number of optimizer steps. Mask refresh and reapplication are explicit, making it possible to distinguish a schedule failure from silent weight regrowth.

Do not execute until the code implements the frozen table above.


In [2]:
torch.manual_seed(SEED)
n,d,c=1400,24,4
x=torch.randn(n,d,device=DEVICE); tw=torch.randn(d,c,device=DEVICE); y=(x@tw+0.25*torch.randn(n,c,device=DEVICE)).argmax(1)
tx,vx=x[:1100],x[1100:]; ty,vy=y[:1100],y[1100:]
def make(): return nn.Sequential(nn.Linear(d,48),nn.ReLU(),nn.Linear(48,c)).to(DEVICE)
def acc(m):
    m.eval();
    with torch.inference_mode(): return float((m(vx).argmax(1)==vy).float().mean().item())
def weights(m): return [p for name,p in m.named_parameters() if "weight" in name]
def set_global_mask(m,rate):
    ps=weights(m); flat=torch.cat([p.detach().abs().flatten() for p in ps]); k=int(flat.numel()*rate); idx=torch.topk(flat,k,largest=False).indices; fm=torch.ones_like(flat); fm[idx]=0
    masks={}; offset=0
    for p in ps: masks[p]=fm[offset:offset+p.numel()].view_as(p); offset+=p.numel()
    with torch.no_grad():
        for p,mask in masks.items(): p.mul_(mask)
    return masks
def train(m,steps,schedule):
    opt=torch.optim.SGD(m.parameters(),lr=0.02,momentum=0.8); masks=None; rows=[]
    for step in range(steps):
        target=schedule(step,steps)
        if step==0 or step%4==0 or step==steps-1: masks=set_global_mask(m,target); rows.append({"step":step,"target":target,"accuracy_before_update":acc(m)})
        idx=torch.arange(step*80,step*80+80,device=DEVICE)%tx.shape[0]; opt.zero_grad(); loss=F.cross_entropy(m(tx[idx]),ty[idx]); loss.backward(); opt.step()
        if masks:
            with torch.no_grad():
                for p,mask in masks.items(): p.mul_(mask)
    return rows,masks
dense=make(); opt=torch.optim.Adam(dense.parameters(),lr=0.03)
for step in range(120):
    idx=torch.arange(step*64,step*64+64,device=DEVICE)%tx.shape[0]; opt.zero_grad(); loss=F.cross_entropy(dense(tx[idx]),ty[idx]); loss.backward(); opt.step()
dense_acc=acc(dense); one=copy.deepcopy(dense); gradual=copy.deepcopy(dense); steps=40
one_rows,one_masks=train(one,steps,lambda step,total:0.80)
def cubic(step,total):
    progress=step/max(total-1,1); return 0.80*(1-(1-progress)**3)
grad_rows,grad_masks=train(gradual,steps,cubic)
one_sp=sum((p==0).sum().item() for p in one_masks)/sum(p.numel() for p in one_masks); grad_sp=sum((p==0).sum().item() for p in grad_masks)/sum(p.numel() for p in grad_masks)
metrics={"dense_accuracy":dense_acc,"oneshot_final_accuracy":acc(one),"gradual_final_accuracy":acc(gradual),"target_sparsity":0.80,"oneshot_actual_sparsity":one_sp,"gradual_actual_sparsity":grad_sp,"schedule_updates":len(grad_rows),"oneshot_trajectory":one_rows,"gradual_trajectory":grad_rows}
analysis=(f"Both routes used {steps} optimizer updates and finished near 80% sparsity. One-shot validation accuracy ended at "
          f"{metrics['oneshot_final_accuracy']:.1%}, while the cubic route ended at {metrics['gradual_final_accuracy']:.1%} "
          f"after {metrics['schedule_updates']} recorded mask updates. The retained trajectories show when each support shock occurred.")


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Dense accuracy | 88.67% |
| One-shot final accuracy | 72.67% |
| Gradual final accuracy | 74.00% |
| Target sparsity | 80.00% |
| Schedule updates | 11 |


## 7. Interpret rather than merely print

Both routes used 40 optimizer updates and finished near 80% sparsity. One-shot validation accuracy ended at 72.7%, while the cubic route ended at 74.0% after 11 recorded mask updates. The retained trajectories show when each support shock occurred.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The tensors and operators executed on CUDA through PyTorch. Native sparse-kernel identity is not inferred unless a trace or backend artifact names it.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 11,
    "title": 'Gradual Pruning Schedules and Recovery Training',
    "environment": ENV,
    "evidence_label": 'pytorch-gpu',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'Gradual pruning controls the sequence of support changes; its endpoint is insufficient to reproduce or judge recovery.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 11,
  "title": "Gradual Pruning Schedules and Recovery Training",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260819
  },
  "evidence_label": "pytorch-gpu",
  "metrics": {
    "dense_accuracy": 0.8866667151451111,
    "oneshot_final_accuracy": 0.7266666889190674,
    "gradual_final_accuracy": 0.7400000095367432,
    "target_sparsity": 0.8,
    "oneshot_actual_sparsity": 0.7998511904761905,
    "gradual_actual_sparsity": 0.7998511904761905,
    "schedule_updates": 11,
    "oneshot_trajectory": [
      {
        "step": 0,
        "target": 0.8,
        "accuracy_before_update": 0.6933333277702332
      },
      {
        "step": 4,
        "target": 0.8,
        "accuracy_before_update": 0.6933333277702332
      },
      {
        "step": 8,
        "target": 0.8,
        "accuracy_before_update": 0.6966666579246521
      },
      {
     

## 9. Make the bounded decision

> Gradual pruning controls the sequence of support changes; its endpoint is insufficient to reproduce or judge recovery.

**Acceptance/rollback:** Accept a schedule only when its complete trajectory, final support, quality recovery, and training budget are recorded and meet the target card.

**Failure analysis:** A schedule can appear better simply because it prunes later and spends more steps near the dense model. Comparing endpoints without integrating the actual sparsity trajectory hides that advantage. Small synthetic data also makes recovery unusually cheap.


## 10. Extend the evidence

Match candidates by area under the sparsity-time curve, sweep update frequency and learning-rate restarts, and repeat on a downstream task metric rather than toy accuracy alone.

The full evidence boundary and references are in [`README.md`](README.md).
